# Online Shoppers Intention Baseline training

## Importing variables and creating variables

In [1]:
import pickle

import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from tqdm.auto import tqdm


/home/mennasalah/online-shoppers-purchasing-intention-ml/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path = "../data/online_shoppers_intention.csv"
categorical = ['visitortype', 'month']
numerical = ['administrative', 'administrative_duration', 'informational',
       'informational_duration', 'productrelated', 'productrelated_duration',
       'bouncerates', 'exitrates', 'pagevalues', 'specialday',
       'operatingsystems', 'browser', 'region', 'traffictype', 'weekend'
       ]
seed = 42

## Data Loading

In [3]:
# loading data
def load(path: str) -> pd.DataFrame:
    df = pd.read_csv(data_path)
    df.columns = df.columns.str.lower()

    return df

In [4]:
## implement
df = load(path=data_path)

# print (X_train.shape, X_test.shape)

In [5]:
df.head()

,administrative,administrative_duration,informational,informational_duration,productrelated,productrelated_duration,bouncerates,exitrates,pagevalues,specialday,month,operatingsystems,browser,region,traffictype,visitortype,weekend,revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [29]:
for col in numerical:
    print(f"max {col}: {max(df[col])}, min: {min(df[col])}")

max administrative: 27, min: 0
max administrative_duration: 3398.75, min: 0.0
max informational: 24, min: 0
max informational_duration: 2549.375, min: 0.0
max productrelated: 705, min: 0
max productrelated_duration: 63973.52223, min: 0.0
max bouncerates: 0.2, min: 0.0
max exitrates: 0.2, min: 0.0
max pagevalues: 361.7637419, min: 0.0
max specialday: 1.0, min: 0.0
max operatingsystems: 8, min: 1
max browser: 13, min: 1
max region: 9, min: 1
max traffictype: 20, min: 1
max weekend: 1, min: 0


## Data Preparation

As per the website and manually checking, there's no NA values,so we don't have to handle that and can get immediately into feature engineering.

In [6]:
# sum is 0
print(df.isna().sum().sum())

0


as an additional experiment we will try both to apply hot one encoding for the `Months` column and to try to have them as numerical data.

In [7]:
def months_names_to_nums(months_column: list[str]) -> list[int]:
    months = {
        "Jan": 1, "Feb": 2, "Mar": 3, "April": 4, "May": 5, "June": 6, "Jul": 7,
        "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
        }
    for month in months_column:
        months_column[month] = int(months[month])

    return months_column

Turn string formatted boolean-value-cloumns (True/False) to integers (0/1)

In [8]:
df['revenue'] = df['revenue'].astype(int)
df['weekend'] = df['weekend'].astype(int)

### Splitting data

I didn't add a val set for now to use Kfold to find best partition to use as val from training data.

In [9]:
def split_df(df: pd.DataFrame):

    # 80/20% split for train and 
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=seed)

    X_train = df_train.reset_index(drop=True)
    X_test = df_test.reset_index(drop=True)


    y_train = X_train.revenue.values
    y_test = X_test.revenue.values

    del X_train['revenue']
    del X_test['revenue']

    return X_train, X_test, y_train, y_test

In [10]:
# df_train is named differently to differentialte it when we split it to train and val sets

df_train, X_test, y_train, y_test = split_df(df)


In [11]:
df_train.head()

,administrative,administrative_duration,informational,informational_duration,productrelated,productrelated_duration,bouncerates,exitrates,pagevalues,specialday,month,operatingsystems,browser,region,traffictype,visitortype,weekend
0,0,0.0,0,0.0,7,95.000000,0.014286,0.061905,0.000000,0.0,Mar,2,6,1,1,Returning_Visitor,0
1,2,14.0,0,0.0,81,1441.910588,0.002469,0.013933,2.769599,0.0,Nov,2,2,3,2,Returning_Visitor,0
2,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.000000,0.0,Mar,2,2,1,1,Returning_Visitor,0
3,5,49.2,4,379.0,5,74.600000,0.000000,0.018182,8.326728,0.0,Sep,2,2,8,2,New_Visitor,0
4,0,0.0,1,5.0,9,279.000000,0.040000,0.041667,0.000000,0.0,Nov,3,2,7,8,New_Visitor,1


In [12]:
def train_model(X_train, y_train, C = 1.0, max_iter = 1000):

    dv = DictVectorizer(sparse=False)

    train_dicts = X_train[categorical + numerical].to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=max_iter)
    model.fit(X_train, y_train)

    return dv, model

In [13]:
def predict(dv: DictVectorizer, model: LogisticRegression, X_test: pd.DataFrame):
    dicts = X_test[categorical + numerical].to_dict(orient='records')
    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred


In [14]:
def cross_validation(df_train, full_y_train):
    kfold = StratifiedKFold(n_splits = 5, shuffle=True, random_state=seed)
    std_dev = {}
    mean_vals = {}
    for C in tqdm([0.000001, 0.001, 1.0, 10.0]):
        auc_scores = []
        # f1_scores = []
        for train_idx, val_idx in kfold.split(df_train, full_y_train):
            X_train = df_train.iloc[train_idx]
            X_val = df_train.iloc[val_idx]

            y_train = full_y_train[train_idx]
            y_val = full_y_train[val_idx]

            dv, model = train_model(X_train, y_train, C=C)


            y_pred = predict(dv=dv, model=model, X_test=X_val)

            auc = roc_auc_score(y_val, y_pred)
            auc_scores.append(auc)
        std_dev[C] = np.std(auc_scores)
        mean_vals[C] = np.mean(auc_scores)
        print("mean value for auc scores:", mean_vals[C])

    return max(mean_vals, key=mean_vals.get)
        


In [15]:
best_C = cross_validation(df_train, y_train)

print("The best C value:", best_C)

 25%|██▌       | 1/4 [00:01<00:04,  1.38s/it]

mean value for auc scores: 0.5043568101658907


 50%|█████     | 2/4 [00:02<00:03,  1.52s/it]

mean value for auc scores: 0.8325635657946538


 75%|███████▌  | 3/4 [00:05<00:01,  1.74s/it]

mean value for auc scores: 0.8969880464949916


100%|██████████| 4/4 [00:06<00:00,  1.70s/it]

mean value for auc scores: 0.8966859225792015
The best C value: 1.0


The C value we found out to be best is **C=1**. This will be the one we will use in all the models we train

## Models Training & Evaluation Metrics

### Baseline Model

In [16]:
# training the basline model - where months are categorical data
dv, model = train_model(df_train, y_train, C = 1.0)

y_pred = predict(dv, model, X_test)

baseline_auc = roc_auc_score(y_test, y_pred)

baseline_auc

0.8963740446717695

In [17]:
def confusion_matrix(y_val, y_pred, t):
    actual_positive = (y_val == 1)
    actual_negative = (y_val == 0)
    
    predict_positive = (y_pred >= t)
    predict_negative = (y_pred < t)

    tp = (predict_positive & actual_positive).sum()
    tn = (predict_negative & actual_negative).sum()

    fp = (predict_positive & actual_negative).sum()
    fn = (predict_negative & actual_positive).sum()

    return tp, tn, fp, fn

In [18]:
tp, tn, fp, fn = confusion_matrix(y_test, y_pred, 0.5)

In [19]:
def calculate_f1_score(tp: int, fp: int, fn: int) -> float:
    """   calculates the f1_score from recall and precision values
    """
    precision = (tp / (tp + fp))
    recall = (tp / (tp + fn))


    if (precision == 0 and recall == 0):
        f1 = 0
    else:
        f1 = 2 * ( (precision * recall) / (precision + recall) ) 

    return f1

In [20]:
baseline_f1 = calculate_f1_score(tp, fp, fn)

baseline_f1

np.float64(0.4785478547854785)

In [21]:
y_pred

array([0.17245456, 0.07666288, 0.2370192 , ..., 0.06433647, 0.36029062,
       0.02547733], shape=(2466,))

In [22]:
y_pred_bin = (y_pred >= 0.5).astype(int)

baeline_accuray = accuracy_score(y_test, y_pred_bin)

baeline_accuray

0.8718572587185726

In [23]:
baseline_model_path= "../models/shopper_intention_model_C=1.bin"

with open(baseline_model_path, "wb") as f_out:
   pickle.dump((dv, model), f_out)

In [24]:
def export_model(model_path, dv, model):

    with open(model_path, "wb") as f_in:
        pickle.dump((dv, model), f_in)

**Baseline model metrics at threshold 0.5:**
- Accuracy: 0.871
- f1: 0.478
- auc: 0.896

---

### Fine-tuning


In [25]:
data_path = "../data/online_shoppers_intention.csv"
categorical_2 = ['visitortype']
numerical_2 = ['administrative', 'administrative_duration', 'informational',
       'informational_duration', 'productrelated', 'productrelated_duration',
       'bouncerates', 'exitrates', 'pagevalues', 'specialday',
       'operatingsystems', 'browser', 'region', 'traffictype', 'weekend', 'month'
       ]

We will try the model without the month being a categorical data and see if it makes a difference in the metrics.

In [26]:
df_months_names_to_nums = df

df_months_names_to_nums['month'] = months_names_to_nums(df_months_names_to_nums['month'])
month_map={
        "Jan": 1, "Feb": 2, "Mar": 3, "April": 4, "May": 5, "June": 6, "Jul": 7,
        "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
        }
df_months_names_to_nums['month'] = df_months_names_to_nums.month.map(month_map)
df_months_names_to_nums.head()


/tmp/ipykernel_10179/1016154935.py:7: Pandas4Warning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. The existing dtype str cannot hold the value being set, so the result has dtype object. Cast the object to object before this operation to retain the current behavior.
  months_column[month] = int(months[month])


,administrative,administrative_duration,informational,informational_duration,productrelated,productrelated_duration,bouncerates,exitrates,pagevalues,specialday,month,operatingsystems,browser,region,traffictype,visitortype,weekend,revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,1,1,1,1,Returning_Visitor,0,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,2,2,2,1,2,Returning_Visitor,0,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,2,4,1,9,3,Returning_Visitor,0,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,2,3,2,2,4,Returning_Visitor,0,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,2,3,3,1,4,Returning_Visitor,1,0


In [27]:
X_train_2, X_test_2, y_train_2, y_test_2 = split_df(df_months_names_to_nums)

dv_2, model_2 = train_model(X_train=X_train_2, y_train=y_train_2, C=1.0)
y_pred_2 = predict(dv_2, model_2, X_test_2)

model_2_auc = roc_auc_score(y_test_2, y_pred_2)
y_pred_bin_2 = (y_pred_2 >= 0.5).astype(int)

model_2_accuray = accuracy_score(y_test_2, y_pred_bin_2)

tp_2, tn_2, fp_2, fn_2 = confusion_matrix(y_test_2, y_pred_2, 0.5)
model_2_f1 = calculate_f1_score(tp_2, fp_2, fn_2)



In [28]:
print(model_2_auc, model_2_f1, model_2_accuray)

0.8914841849148418 0.453781512605042 0.8682076236820763


So we decide that having the months as categorical data with one hot encoded values gives better metrics for the model

---

In [31]:
df.columns

Index(['administrative', 'administrative_duration', 'informational',
       'informational_duration', 'productrelated', 'productrelated_duration',
       'bouncerates', 'exitrates', 'pagevalues', 'specialday', 'month',
       'operatingsystems', 'browser', 'region', 'traffictype', 'visitortype',
       'weekend', 'revenue'],
      dtype='str')